# Differential Equations — Session 10
## Section 3.1: Linear Models

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to:

1. formulate a first-order model from assumptions and units;
2. use exponential growth and decay models;
3. derive doubling-time and half-life formulas;
4. formulate Newton cooling and warming;
5. construct mixing equations using rate in minus rate out;
6. distinguish constant-volume and variable-volume tanks;
7. formulate an LR-circuit model;
8. interpret transient and steady-state behavior.

> The notebook preserves the main theoretical structure of Section 3.1 while using original examples, diagrams, and simulations.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–12 min | Modeling assumptions, units, and balance laws |
| 12–28 min | Growth, decay, doubling time, and half-life |
| 28–43 min | Newton cooling/warming |
| 43–67 min | Mixing tanks |
| 67–82 min | LR circuits and transient response |
| 82–90 min | Synthesis and exit check |

The carbon-dating and variable-volume simulations can be used as extensions or assigned for independent exploration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
from matplotlib.patches import Rectangle, FancyArrowPatch, Circle
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=5, suppress=True)

def safe_solve(rhs, t_span, y0, points=700, **kwargs):
    t_eval = np.linspace(t_span[0], t_span[1], points)
    return solve_ivp(rhs, t_span, y0, t_eval=t_eval, **kwargs)

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 3.1-A — Mathematical model

A **mathematical model** is a mathematical description of a system constructed from:

- state variables,
- parameters,
- initial conditions,
- empirical or physical laws,
- and explicit simplifying assumptions.

### Principle 3.1-B — Balance law

Many first-order models have the structure

$$
\frac{d}{dt}(\text{amount})
=
\text{rate in}
-
\text{rate out}
+
\text{rate produced}
-
\text{rate lost}.
$$

Every term must have the same physical units.

### Theorem 3.1-C — Exponential growth or decay

For

$$
A'=kA,
\qquad
A(0)=A_0,
$$

the unique solution is

$$
A(t)=A_0e^{kt}.
$$

If $k>0$, the quantity grows. If $k<0$, it decays.

### Corollary 3.1-D — Doubling time and half-life

For $k>0$, the doubling time is

$$
T_d=\frac{\ln 2}{k}.
$$

For $k<0$, the half-life is

$$
T_{1/2}=\frac{\ln 2}{|k|}.
$$

### Theorem 3.1-E — Newton cooling/warming

If the ambient temperature $T_m$ is constant and

$$
T'=-k(T-T_m),
\qquad
k>0,
$$

then

$$
T(t)=T_m+(T_0-T_m)e^{-kt}.
$$

The object approaches $T_m$ asymptotically and does not reach it at a finite time unless $T_0=T_m$.

### Principle 3.1-F — Well-mixed tank

If $A(t)$ is the amount of solute and $V(t)$ is the liquid volume, then

$$
A'
=
r_{\text{in}}c_{\text{in}}
-
r_{\text{out}}\frac{A}{V(t)}.
$$

The model assumes instantaneous uniform mixing.

### Theorem 3.1-G — Constant-voltage LR circuit

For

$$
L i'+Ri=E_0,
\qquad
i(0)=i_0,
$$

the current is

$$
i(t)=\frac{E_0}{R}
+
\left(i_0-\frac{E_0}{R}\right)e^{-Rt/L}.
$$

The first term is the steady-state current and the exponential term is the transient.

### Classroom Checkpoint — Units in a Mixing Model

In a mixing equation $A'=\text{rate in}-\text{rate out}$, what units must every term have?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Modeling begins before solving

For each application, identify:

1. the state variable and units;
2. the independent variable and units;
3. all parameters and their units;
4. the mechanism that creates each rate term;
5. the interval on which the assumptions remain valid.

A correct differential equation with poorly defined units is not yet a complete model.

## 2. Exponential growth and decay

Suppose a cell culture begins with 250 cells and increases by 40% during the first two hours. Under the proportional-growth assumption,

$$
P'=kP,
\qquad
P(0)=250.
$$

The observation $P(2)=350$ gives

$$
e^{2k}=1.4,
\qquad
k=\frac{\ln(1.4)}{2}.
$$

In [ ]:
P0 = 250
k = np.log(1.4)/2
doubling_time = np.log(2)/k

t = np.linspace(0, 12, 500)
P = P0*np.exp(k*t)

plt.plot(t, P, linewidth=2)
plt.scatter([0, 2], [250, 350], s=70)
plt.axvline(doubling_time, linestyle="--", label=f"doubling time = {doubling_time:.2f} h")
plt.xlabel("time (hours)")
plt.ylabel("population")
plt.title("Calibrating an exponential growth model")
plt.legend()
plt.show()

print("Estimated growth constant:", k, "per hour")
print("Doubling time:", doubling_time, "hours")

### Interactive growth/decay experiment

The same equation describes both growth and decay. The sign and magnitude of $k$ control the direction and time scale.

In [ ]:
def exponential_explorer(A0=100.0, k=0.2, final_time=20):
    t = np.linspace(0, final_time, 600)
    A = A0*np.exp(k*t)

    plt.plot(t, A, linewidth=2)
    plt.scatter([0], [A0], s=70)
    plt.xlabel("time")
    plt.ylabel("A(t)")
    plt.title(fr"$A'=kA$ with $k={k:.3f}$")
    plt.show()

    if k > 0:
        print("Doubling time:", np.log(2)/k)
    elif k < 0:
        print("Half-life:", np.log(2)/abs(k))
    else:
        print("Constant solution.")

if WIDGETS_AVAILABLE:
    interact(
        exponential_explorer,
        A0=FloatSlider(min=10, max=500, step=10, value=100),
        k=FloatSlider(min=-0.5, max=0.5, step=0.025, value=0.2),
        final_time=IntSlider(min=5, max=50, step=5, value=20)
    )
else:
    exponential_explorer()

## 3. Radiometric dating as inverse modeling

If a radioactive isotope has half-life $H$, then

$$
A(t)=A_0\,2^{-t/H}.
$$

If the observed fraction remaining is $q=A(t)/A_0$, then

$$
t=-H\frac{\ln q}{\ln 2}.
$$

The model estimates time from a measured present-day fraction.

In [ ]:
def dating_explorer(fraction=0.25, half_life=5730):
    age = -half_life*np.log(fraction)/np.log(2)

    fractions = np.logspace(-4, 0, 500)
    ages = -half_life*np.log(fractions)/np.log(2)

    plt.semilogx(fractions, ages, linewidth=2)
    plt.scatter([fraction], [age], s=70)
    plt.xlabel("fraction remaining")
    plt.ylabel("estimated age")
    plt.title("Radiometric age as an inverse exponential calculation")
    plt.show()

    print("Estimated age:", age)

if WIDGETS_AVAILABLE:
    interact(
        dating_explorer,
        fraction=FloatSlider(min=0.001, max=1.0, step=0.001, value=0.25),
        half_life=IntSlider(min=500, max=20000, step=100, value=5730)
    )
else:
    dating_explorer()

## 4. Newton cooling and warming

A metal object at $95^\circ\text{C}$ is placed in a room at $22^\circ\text{C}$. After 8 minutes its temperature is $60^\circ\text{C}$.

The model is

$$
T'=-k(T-22),
\qquad
T(0)=95.
$$

From $T(8)=60$,

$$
60-22=(95-22)e^{-8k}.
$$

In [ ]:
T0, Tm, T8 = 95.0, 22.0, 60.0
k_cool = -(1/8)*np.log((T8-Tm)/(T0-Tm))

t = np.linspace(0, 50, 600)
T = Tm + (T0-Tm)*np.exp(-k_cool*t)

plt.plot(t, T, linewidth=2, label="object temperature")
plt.axhline(Tm, linestyle="--", label="ambient temperature")
plt.scatter([0, 8], [T0, T8], s=70)
plt.xlabel("time (minutes)")
plt.ylabel("temperature (C)")
plt.title("Newton cooling model")
plt.legend()
plt.show()

print("Cooling constant:", k_cool, "per minute")
print("Temperature after 20 minutes:", Tm+(T0-Tm)*np.exp(-k_cool*20))

### Interactive interpretation

A practical question is often not “When does the object equal the room temperature?” but “When is it within a chosen tolerance?”

In [ ]:
def cooling_explorer(T0=90.0, Tm=22.0, k=0.12, tolerance=1.0):
    t = np.linspace(0, 60, 700)
    T = Tm+(T0-Tm)*np.exp(-k*t)
    target_time = np.log(abs(T0-Tm)/tolerance)/k if tolerance < abs(T0-Tm) else 0

    plt.plot(t, T, linewidth=2)
    plt.axhline(Tm, linestyle="--", label="ambient")
    plt.axhline(Tm+tolerance, linestyle=":")
    plt.axhline(Tm-tolerance, linestyle=":")
    plt.axvline(target_time, linestyle="--", label=f"within tolerance at t≈{target_time:.2f}")
    plt.xlabel("time")
    plt.ylabel("temperature")
    plt.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        cooling_explorer,
        T0=FloatSlider(min=0, max=120, step=2, value=90),
        Tm=FloatSlider(min=0, max=50, step=1, value=22),
        k=FloatSlider(min=0.02, max=0.5, step=0.02, value=0.12),
        tolerance=FloatSlider(min=0.5, max=10, step=0.5, value=1)
    )
else:
    cooling_explorer()

## 5. Mixing tanks: derive before solving

Let $A(t)$ be grams of salt and $V(t)$ be liters of liquid.

The concentration in the tank is

$$
\frac{A(t)}{V(t)}.
$$

The governing equation is

$$
A'
=
(\text{inflow rate})(\text{inflow concentration})
-
(\text{outflow rate})(\text{tank concentration}).
$$

In [ ]:
# Original schematic diagram for a well-mixed tank
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.axis("off")

tank = Rectangle((0.35, 0.15), 0.3, 0.55, fill=False, linewidth=2)
ax.add_patch(tank)
ax.text(0.50, 0.43, r"Well-mixed tank" + "\n" + r"$A(t)$ grams, $V(t)$ liters",
        ha="center", va="center")

ax.add_patch(FancyArrowPatch((0.08, 0.58), (0.35, 0.58), arrowstyle="->", mutation_scale=18))
ax.add_patch(FancyArrowPatch((0.65, 0.30), (0.92, 0.30), arrowstyle="->", mutation_scale=18))

ax.text(0.20, 0.66, r"$r_{\rm in},\,c_{\rm in}$", ha="center")
ax.text(0.80, 0.38, r"$r_{\rm out},\,A/V$", ha="center")
ax.text(0.50, 0.03, r"$A'=r_{\rm in}c_{\rm in}-r_{\rm out}A/V$", ha="center")
plt.show()

### Constant-volume tank

A 150-L tank initially contains 30 g of salt. Brine with concentration $1.2$ g/L enters and leaves at 5 L/min.

Then

$$
A'=6-\frac{A}{30},
\qquad
A(0)=30.
$$

The equilibrium amount is

$$
A^*=Vc_{\text{in}}=180\text{ g}.
$$

In [ ]:
def constant_tank(A0=30.0, V=150.0, r=5.0, c_in=1.2, final_time=150):
    def rhs(t, A):
        return [r*c_in-r*A[0]/V]

    sol = safe_solve(rhs, (0, final_time), [A0])
    equilibrium = V*c_in

    plt.plot(sol.t, sol.y[0], linewidth=2, label="salt amount")
    plt.axhline(equilibrium, linestyle="--", label="equilibrium")
    plt.xlabel("time (minutes)")
    plt.ylabel("salt (grams)")
    plt.title("Constant-volume mixing model")
    plt.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        constant_tank,
        A0=FloatSlider(min=0, max=400, step=10, value=30),
        V=FloatSlider(min=50, max=300, step=10, value=150),
        r=FloatSlider(min=1, max=15, step=1, value=5),
        c_in=FloatSlider(min=0, max=3, step=0.1, value=1.2),
        final_time=IntSlider(min=30, max=300, step=30, value=150)
    )
else:
    constant_tank()

### Variable-volume tank

If inflow and outflow rates differ, then

$$
V(t)=V_0+(r_{\text{in}}-r_{\text{out}})t.
$$

The model is valid only while the tank has positive volume and has not overflowed.

In [ ]:
def variable_tank(A0=40.0, V0=100.0, rin=4.0, rout=2.0, c_in=1.0, capacity=180.0):
    net = rin-rout
    if net > 0 and capacity <= V0:
        print("Choose a capacity larger than the initial volume.")
        return
    if net > 0:
        final_time = 0.98*(capacity-V0)/net
    elif net < 0:
        final_time = 0.98*V0/(-net)
    else:
        final_time = 80

    def rhs(t, A):
        V = V0+net*t
        return [rin*c_in-rout*A[0]/V]

    sol = safe_solve(rhs, (0, max(final_time, 1e-3)), [A0])
    V = V0+net*sol.t

    plt.plot(sol.t, sol.y[0], label="salt amount A(t)")
    plt.plot(sol.t, V, linestyle="--", label="liquid volume V(t)")
    plt.xlabel("time")
    plt.ylabel("amount / volume")
    plt.title("Variable-volume mixing model")
    plt.legend()
    plt.show()

    print("Displayed model end time:", final_time)
    if net > 0:
        print("Overflow time:", (capacity-V0)/net)
    elif net < 0:
        print("Emptying time:", V0/(-net))

if WIDGETS_AVAILABLE:
    interact(
        variable_tank,
        A0=FloatSlider(min=0, max=200, step=10, value=40),
        V0=FloatSlider(min=40, max=150, step=10, value=100),
        rin=FloatSlider(min=1, max=8, step=0.5, value=4),
        rout=FloatSlider(min=1, max=8, step=0.5, value=2),
        c_in=FloatSlider(min=0, max=3, step=0.1, value=1),
        capacity=FloatSlider(min=120, max=300, step=10, value=180)
    )
else:
    variable_tank()

## 6. LR-series circuit

Kirchhoff's voltage law gives

$$
L\frac{di}{dt}+Ri=E(t).
$$

For a constant source $E_0$, the time constant is

$$
\tau=\frac{L}{R}.
$$

After one time constant, about $63.2\%$ of the transition toward steady state has occurred.

In [ ]:
def lr_explorer(L=0.8, R=6.0, E0=12.0, i0=0.0):
    tau = L/R
    t = np.linspace(0, 6*tau, 600)
    steady = E0/R
    i = steady+(i0-steady)*np.exp(-t/tau)

    plt.plot(t, i, linewidth=2, label="current")
    plt.axhline(steady, linestyle="--", label="steady-state current")
    plt.axvline(tau, linestyle=":", label=f"time constant={tau:.3f}")
    plt.xlabel("time (seconds)")
    plt.ylabel("current (amperes)")
    plt.title("Transient and steady-state current in an LR circuit")
    plt.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        lr_explorer,
        L=FloatSlider(min=0.1, max=3.0, step=0.1, value=0.8),
        R=FloatSlider(min=1, max=20, step=1, value=6),
        E0=FloatSlider(min=1, max=30, step=1, value=12),
        i0=FloatSlider(min=-3, max=5, step=0.25, value=0)
    )
else:
    lr_explorer()

## Model synthesis

| Application | State variable | Governing mechanism | Long-term behavior |
|---|---|---|---|
| Growth/decay | amount or population | proportional rate | unbounded growth or decay to zero |
| Cooling | temperature difference | proportional relaxation | approaches ambient temperature |
| Mixing | solute amount | rate in minus rate out | approaches inflow concentration when volume is constant |
| LR circuit | current | voltage balance | approaches $E_0/R$ |

The formulas differ, but each model contains a state, a rate law, parameters, initial data, and a domain of validity.

## Classroom Checkpoint — Exit Check

A 200-L tank receives brine at 3 L/min with concentration 2 g/L and loses mixture at the same rate.

1. What is the state variable?
2. Write the differential equation.
3. What is the equilibrium salt amount?
4. What assumption justifies the output concentration?

> Pause here. Let students commit to an answer before running the next cell.